# E-Commerce Analytics Project
## 04 — SQL Business Analysis

### Objective
Use SQL to query the cleaned e-commerce database, reproduce key business findings, and demonstrate relational analysis skills using joins, aggregations, CTEs, date functions, and window functions.

### Topics Covered
- SQL connection and table inspection
- Revenue and profit analysis
- Product and category performance
- Returns analysis
- Customer behavior
- Marketing and acquisition analysis
- Geographic performance
- Monthly growth using window functions
- Customer ranking

In [1]:
import pandas as pd
import sqlite3

from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATABASE_PATH = PROJECT_ROOT / "database" / "ecommerce.db"

print("Database:", DATABASE_PATH)
print("Exists:", DATABASE_PATH.exists())

Database: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/database/ecommerce.db
Exists: True


In [3]:
connection = sqlite3.connect(DATABASE_PATH)

print("Connected to SQLite database.")

Connected to SQLite database.


In [4]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

tables

,name
0,clean_customers
1,clean_marketing
2,clean_order_items
3,clean_orders
4,clean_products
5,clean_returns
6,raw_customers
7,raw_marketing
8,raw_order_items
9,raw_orders


## 1. Annual Revenue Performance

The first query calculates annual revenue from completed orders using the cleaned Orders and Order Items tables.

In [5]:
annual_revenue_query = """
SELECT
    CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ),
        2
    ) AS revenue,

    COUNT(DISTINCT o.order_id) AS orders,

    COUNT(DISTINCT o.customer_id) AS customers

FROM clean_order_items AS oi

INNER JOIN clean_orders AS o
    ON oi.order_id = o.order_id

WHERE o.order_status = 'Completed'

GROUP BY year

ORDER BY year;
"""

annual_revenue_sql = pd.read_sql_query(
    annual_revenue_query,
    connection
)

annual_revenue_sql

,year,revenue,orders,customers
0,2023,7497268.62,20497,6320
1,2024,8788323.82,24188,11792
2,2025,10210293.94,28071,16149


In [6]:
annual_growth_query = """
WITH annual_sales AS (

    SELECT
        CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ) AS revenue

    FROM clean_order_items AS oi

    INNER JOIN clean_orders AS o
        ON oi.order_id = o.order_id

    WHERE o.order_status = 'Completed'

    GROUP BY year
),

growth_analysis AS (

    SELECT
        year,
        revenue,

        LAG(revenue) OVER (
            ORDER BY year
        ) AS prior_year_revenue

    FROM annual_sales
)

SELECT
    year,

    ROUND(revenue, 2) AS revenue,

    ROUND(
        (revenue - prior_year_revenue)
        / prior_year_revenue
        * 100,
        2
    ) AS yoy_growth_pct

FROM growth_analysis

ORDER BY year;
"""

In [7]:
annual_growth_sql = pd.read_sql_query(
    annual_growth_query,
    connection
)

annual_growth_sql

,year,revenue,yoy_growth_pct
0,2023,7497268.62,NaN
1,2024,8788323.82,17.22
2,2025,10210293.94,16.18


## 2. Category Performance

This query joins Orders, Order Items, and Products to evaluate revenue, gross profit, and gross margin by category.

In [8]:
category_query = """
SELECT
    p.category,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ),
        2
    ) AS revenue,

    ROUND(
        SUM(
            (
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_pct)
            )
            -
            (
                oi.quantity
                * p.unit_cost
            )
        ),
        2
    ) AS gross_profit,

    ROUND(
        SUM(
            (
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_pct)
            )
            -
            (
                oi.quantity
                * p.unit_cost
            )
        )
        /
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        )
        * 100,
        2
    ) AS gross_margin_pct,

    SUM(oi.quantity) AS units_sold

FROM clean_order_items AS oi

INNER JOIN clean_orders AS o
    ON oi.order_id = o.order_id

INNER JOIN clean_products AS p
    ON oi.product_id = p.product_id

WHERE o.order_status = 'Completed'

GROUP BY p.category

ORDER BY revenue DESC;
"""

In [9]:
category_sql = pd.read_sql_query(
    category_query,
    connection
)

category_sql

,category,revenue,gross_profit,gross_margin_pct,units_sold
0,Electronics,6846336.21,3035091.14,44.33,39337
1,Home & Kitchen,6175535.52,2648837.76,42.89,39322
2,Beauty & Health,5886739.53,2495887.62,42.40,33242
3,Sports & Outdoors,3844103.68,1594172.16,41.47,24250
4,Fashion,3743171.43,1519670.63,40.60,32150


## 3. Returns and Refund Impact

A LEFT JOIN is used to preserve all completed order items while identifying which items were returned.

This allows us to calculate return rates, refunds, and revenue retained after refunds by product category.

In [10]:
returns_query = """
WITH transaction_returns AS (

    SELECT
        oi.order_item_id,
        p.category,

        (
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ) AS revenue,

        COALESCE(r.refund_amount, 0) AS refund_amount,

        CASE
            WHEN r.return_id IS NOT NULL THEN 1
            ELSE 0
        END AS was_returned

    FROM clean_order_items AS oi

    INNER JOIN clean_orders AS o
        ON oi.order_id = o.order_id

    INNER JOIN clean_products AS p
        ON oi.product_id = p.product_id

    LEFT JOIN clean_returns AS r
        ON oi.order_item_id = r.order_item_id

    WHERE o.order_status = 'Completed'
)

SELECT
    category,

    COUNT(*) AS items_sold,

    SUM(was_returned) AS returned_items,

    ROUND(
        SUM(was_returned) * 100.0
        / COUNT(*),
        2
    ) AS return_rate_pct,

    ROUND(
        SUM(revenue),
        2
    ) AS gross_revenue,

    ROUND(
        SUM(refund_amount),
        2
    ) AS refunds,

    ROUND(
        SUM(revenue)
        - SUM(refund_amount),
        2
    ) AS revenue_after_refunds

FROM transaction_returns

GROUP BY category

ORDER BY return_rate_pct DESC;
"""

In [11]:
returns_sql = pd.read_sql_query(
    returns_query,
    connection
)

returns_sql

,category,items_sold,returned_items,return_rate_pct,gross_revenue,refunds,revenue_after_refunds
0,Fashion,23225,3418,14.72,3743171.43,542210.98,3200960.45
1,Electronics,28570,2283,7.99,6846336.21,548449.37,6297886.84
2,Sports & Outdoors,17540,1394,7.95,3844103.68,298040.67,3546063.01
3,Home & Kitchen,28301,1973,6.97,6175535.52,437941.63,5737593.89
4,Beauty & Health,23889,1201,5.03,5886739.53,296954.44,5589785.09


## 4. Customer Retention Analysis

Customers are classified as One-Time or Returning using SQL aggregation and CASE WHEN logic.

In [12]:
customer_type_query = """
WITH customer_summary AS (

    SELECT
        o.customer_id,

        COUNT(
            DISTINCT o.order_id
        ) AS orders,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ) AS revenue

    FROM clean_orders AS o

    INNER JOIN clean_order_items AS oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'Completed'

    GROUP BY o.customer_id
),

classified_customers AS (

    SELECT
        customer_id,
        orders,
        revenue,

        CASE
            WHEN orders > 1
                THEN 'Returning'
            ELSE 'One-Time'
        END AS customer_type

    FROM customer_summary
)

SELECT
    customer_type,

    COUNT(*) AS customers,

    SUM(orders) AS total_orders,

    ROUND(
        SUM(revenue),
        2
    ) AS revenue,

    ROUND(
        AVG(revenue),
        2
    ) AS revenue_per_customer

FROM classified_customers

GROUP BY customer_type

ORDER BY revenue DESC;
"""

In [13]:
customer_type_sql = pd.read_sql_query(
    customer_type_query,
    connection
)

customer_type_sql

,customer_type,customers,total_orders,revenue,revenue_per_customer
0,Returning,14800,68035,24748831.67,1672.22
1,One-Time,4721,4721,1747054.72,370.06


## 5. Product Revenue Ranking

Products are ranked by revenue using the SQL RANK() window function.

In [14]:
product_ranking_query = """
WITH product_sales AS (

    SELECT
        p.product_id,
        p.product_name,
        p.category,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ) AS revenue,

        SUM(
            (
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_pct)
            )
            -
            (
                oi.quantity
                * p.unit_cost
            )
        ) AS gross_profit

    FROM clean_order_items AS oi

    INNER JOIN clean_orders AS o
        ON oi.order_id = o.order_id

    INNER JOIN clean_products AS p
        ON oi.product_id = p.product_id

    WHERE o.order_status = 'Completed'

    GROUP BY
        p.product_id,
        p.product_name,
        p.category
)

SELECT
    product_id,
    product_name,
    category,

    ROUND(
        revenue,
        2
    ) AS revenue,

    ROUND(
        gross_profit,
        2
    ) AS gross_profit,

    RANK() OVER (
        ORDER BY revenue DESC
    ) AS revenue_rank,

    RANK() OVER (
        ORDER BY gross_profit DESC
    ) AS profit_rank

FROM product_sales

ORDER BY revenue_rank

LIMIT 10;
"""

In [15]:
product_ranking_sql = pd.read_sql_query(
    product_ranking_query,
    connection
)

product_ranking_sql

,product_id,product_name,category,revenue,gross_profit,revenue_rank,profit_rank
0,PROD-0050,Webcam 50,Electronics,538231.73,285296.90,1,1
1,PROD-0114,Webcam 114,Electronics,531809.60,272794.88,2,2
2,PROD-0104,Headphones 104,Electronics,487911.33,226145.67,3,6
3,PROD-0043,Smart Watch 43,Electronics,487650.24,248175.20,4,3
4,PROD-0012,Webcam 12,Electronics,470019.76,246385.36,5,4
5,PROD-0079,Webcam 79,Electronics,449726.15,236867.27,6,5
6,PROD-0033,Sneakers 33,Fashion,425700.71,221501.32,7,7
7,PROD-0053,Desk Lamp 53,Home & Kitchen,408048.06,197050.35,8,9
8,PROD-0042,Bluetooth Speaker 42,Electronics,390087.75,185022.75,9,11
9,PROD-0109,Hair Dryer 109,Beauty & Health,379703.52,204854.07,10,8


## 6. Monthly Revenue Growth

Monthly revenue is analyzed using a SQL window function to calculate month-over-month growth.

This demonstrates the use of:
- CTEs
- Date aggregation
- LAG()
- Window functions

In [16]:
monthly_growth_query = """
WITH monthly_sales AS (

    SELECT
        strftime(
            '%Y-%m',
            o.order_date
        ) AS month,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ) AS revenue

    FROM clean_orders AS o

    INNER JOIN clean_order_items AS oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'Completed'

    GROUP BY month
),

monthly_comparison AS (

    SELECT
        month,
        revenue,

        LAG(revenue) OVER (
            ORDER BY month
        ) AS prior_month_revenue,

        LAG(revenue, 12) OVER (
            ORDER BY month
        ) AS prior_year_revenue

    FROM monthly_sales
)

SELECT
    month,

    ROUND(
        revenue,
        2
    ) AS revenue,

    ROUND(
        (
            revenue - prior_month_revenue
        )
        / prior_month_revenue
        * 100,
        2
    ) AS mom_growth_pct,

    ROUND(
        (
            revenue - prior_year_revenue
        )
        / prior_year_revenue
        * 100,
        2
    ) AS yoy_growth_pct

FROM monthly_comparison

ORDER BY month;
"""

In [17]:
monthly_growth_sql = pd.read_sql_query(
    monthly_growth_query,
    connection
)

monthly_growth_sql

,month,revenue,mom_growth_pct,yoy_growth_pct
0,2023-01,467865.89,NaN,NaN
1,2023-02,414288.87,-11.45,NaN
2,2023-03,567475.84,36.98,NaN
3,2023-04,583882.91,2.89,NaN
4,2023-05,585248.46,0.23,NaN
5,2023-06,596095.17,1.85,NaN
6,2023-07,647599.67,8.64,NaN
7,2023-08,618996.87,-4.42,NaN
8,2023-09,587518.19,-5.09,NaN
9,2023-10,621521.49,5.79,NaN


## 7. Geographic Performance

Customer, order, and transaction data are combined to evaluate revenue and profitability across geographic regions.

In [18]:
geography_query = """
SELECT
    c.region,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        ),
        2
    ) AS revenue,

    ROUND(
        SUM(
            (
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_pct)
            )
            -
            (
                oi.quantity
                * p.unit_cost
            )
        ),
        2
    ) AS gross_profit,

    COUNT(
        DISTINCT o.order_id
    ) AS orders,

    COUNT(
        DISTINCT o.customer_id
    ) AS customers,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        )
        /
        COUNT(
            DISTINCT o.order_id
        ),
        2
    ) AS aov

FROM clean_orders AS o

INNER JOIN clean_order_items AS oi
    ON o.order_id = oi.order_id

INNER JOIN clean_products AS p
    ON oi.product_id = p.product_id

INNER JOIN clean_customers AS c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'Completed'

GROUP BY c.region

ORDER BY revenue DESC;
"""

In [19]:
geography_sql = pd.read_sql_query(
    geography_query,
    connection
)

geography_sql

,region,revenue,gross_profit,orders,customers,aov
0,West,7979375.50,3394513.50,21973,5843,363.14
1,Midwest,6658741.09,2838788.61,18263,4909,364.60
2,South,6508344.14,2783257.32,17875,4832,364.10
3,Northeast,5349425.64,2277099.88,14645,3937,365.27


## 8. Master Analytical View

A reusable SQL view is created to combine customer, order, product, return, and financial information into a single analytical structure.

This view can support downstream business analysis and dashboard development.

In [20]:
create_view_query = """
DROP VIEW IF EXISTS analytics_transactions;
"""

connection.execute(create_view_query)

In [21]:
create_view_query = """
CREATE VIEW analytics_transactions AS

SELECT
    oi.order_item_id,
    o.order_id,
    o.order_date,

    o.customer_id,
    c.city,
    c.state,
    c.region,
    c.age_group,
    c.acquisition_channel,

    o.device,
    o.sales_channel,
    o.shipping_cost,
    o.order_status,

    p.product_id,
    p.product_name,
    p.category,
    p.subcategory,

    oi.quantity,
    oi.unit_price,
    oi.discount_pct,

    ROUND(
        oi.quantity
        * oi.unit_price,
        2
    ) AS gross_sales,

    ROUND(
        oi.quantity
        * oi.unit_price
        * oi.discount_pct,
        2
    ) AS discount_amount,

    ROUND(
        oi.quantity
        * oi.unit_price
        * (1 - oi.discount_pct),
        2
    ) AS revenue,

    ROUND(
        oi.quantity
        * p.unit_cost,
        2
    ) AS cogs,

    ROUND(
        (
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        )
        -
        (
            oi.quantity
            * p.unit_cost
        ),
        2
    ) AS gross_profit,

    CASE
        WHEN r.return_id IS NOT NULL
            THEN 1
        ELSE 0
    END AS was_returned,

    COALESCE(
        r.refund_amount,
        0
    ) AS refund_amount,

    ROUND(
        (
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_pct)
        )
        -
        COALESCE(
            r.refund_amount,
            0
        ),
        2
    ) AS revenue_after_refunds

FROM clean_order_items AS oi

INNER JOIN clean_orders AS o
    ON oi.order_id = o.order_id

INNER JOIN clean_products AS p
    ON oi.product_id = p.product_id

INNER JOIN clean_customers AS c
    ON o.customer_id = c.customer_id

LEFT JOIN clean_returns AS r
    ON oi.order_item_id = r.order_item_id;
"""

connection.execute(create_view_query)

connection.commit()

print(
    "analytics_transactions view created."
)

analytics_transactions view created.


In [22]:
master_preview = pd.read_sql_query(
    """
    SELECT *
    FROM analytics_transactions
    LIMIT 10;
    """,
    connection
)

master_preview

,order_item_id,order_id,order_date,customer_id,city,state,region,age_group,acquisition_channel,device,...,unit_price,discount_pct,gross_sales,discount_amount,revenue,cogs,gross_profit,was_returned,refund_amount,revenue_after_refunds
0,ITEM-0000001,ORD-000001,2025-03-05 00:00:00,CUST-20202,Port Gabriel,TX,South,25-34,Affiliate,Desktop,...,225.92,0.30,225.92,67.78,158.14,149.15,8.99,0,0.00,158.14
1,ITEM-0000002,ORD-000001,2025-03-05 00:00:00,CUST-20202,Port Gabriel,TX,South,25-34,Affiliate,Desktop,...,233.26,0.05,233.26,11.66,221.60,126.19,95.41,0,0.00,221.60
2,ITEM-0000003,ORD-000001,2025-03-05 00:00:00,CUST-20202,Port Gabriel,TX,South,25-34,Affiliate,Desktop,...,31.18,0.00,31.18,0.00,31.18,22.88,8.30,0,0.00,31.18
3,ITEM-0000004,ORD-000002,2024-12-05 00:00:00,CUST-18773,Davidfurt,PA,Northeast,45-54,Social Media,Desktop,...,302.76,0.30,302.76,90.83,211.93,130.02,81.91,0,0.00,211.93
4,ITEM-0000005,ORD-000002,2024-12-05 00:00:00,CUST-18773,Davidfurt,PA,Northeast,45-54,Social Media,Desktop,...,47.22,0.00,94.44,0.00,94.44,53.32,41.12,1,94.44,0.00
5,ITEM-0000006,ORD-000003,2025-04-21 00:00:00,CUST-16907,Houseside,AZ,West,45-54,Affiliate,Mobile,...,251.16,0.00,251.16,0.00,251.16,108.90,142.26,0,0.00,251.16
6,ITEM-0000007,ORD-000004,2024-05-10 00:00:00,CUST-12557,Lake Jeffrey,CO,West,35-44,Organic Search,Mobile,...,309.23,0.00,309.23,0.00,309.23,128.85,180.38,0,0.00,309.23
7,ITEM-0000008,ORD-000004,2024-05-10 00:00:00,CUST-12557,Lake Jeffrey,CO,West,35-44,Organic Search,Mobile,...,129.41,0.10,129.41,12.94,116.47,58.53,57.94,0,0.00,116.47
8,ITEM-0000009,ORD-000005,2025-06-01 00:00:00,CUST-01431,West Bryan,AZ,West,45-54,Direct,Tablet,...,36.00,0.00,36.00,0.00,36.00,22.31,13.69,0,0.00,36.00
9,ITEM-0000010,ORD-000005,2025-06-01 00:00:00,CUST-01431,West Bryan,AZ,West,45-54,Direct,Tablet,...,146.11,0.30,438.33,131.50,306.83,308.28,-1.45,0,0.00,306.83


In [23]:
pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS rows
    FROM analytics_transactions;
    """,
    connection
)

,rows
0,125273
